# 04 — Robustness and out-of-time prediction

**What we are doing:** Ask whether the distance coefficient changes if we remove ZCTA 30331 or drop cells without reported origin records. Then train a simple destination-share model on Jan–Sep and test on Oct–Dec. The historical-share benchmark is important: a complicated model should beat a simple baseline before claiming predictive value.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
PROCESSED=ROOT/'data'/'processed'
formula='flow ~ log_distance + C(ZCTA) + C(Mall) + C(Month)'
rows=[]
for scenario in ['2020_geometry_provisional','2010_equal_block_diagnostic']:
    p=pd.read_csv(PROCESSED/f'panel_{scenario}.csv',dtype={'ZCTA':str})
    for sample,sub in [('All 25',p),('Original 24',p[p.ZCTA!='30331']),('Reported only',p[~p.unreported_origin_cell])]:
        model=smf.glm(formula,data=sub,family=sm.families.Poisson())
        groups=sub.loc[model.data.row_labels,'ZCTA']
        fit=model.fit(cov_type='cluster',cov_kwds={'groups':groups},maxiter=150)
        rows.append({'scenario':scenario,'sample':sample,'beta_distance':fit.params['log_distance'],'se':fit.bse['log_distance'],'n':fit.nobs})
robust=pd.DataFrame(rows)
print(robust.to_string(index=False))

## Train/test destination shares
**What we are doing:** Fit a Poisson model on Jan–Sep. Predict Oct–Dec and normalize the four destination predictions to shares within each origin-month. Compare mean absolute share error with each origin’s Jan–Sep historical shares.

$$MAE_{pp}=100\times\frac{1}{N}\sum|\hat{s}_{ijt}-s_{ijt}|$$

In [ ]:
rows=[]
for scenario in ['2020_geometry_provisional','2010_equal_block_diagnostic']:
    p=pd.read_csv(PROCESSED/f'panel_{scenario}.csv',dtype={'ZCTA':str})
    p=p.dropna(subset=['income','population']).copy()
    train=p[p.Month<=9].copy(); test=p[p.Month>=10].copy()
    history=train.groupby(['ZCTA','Mall'],as_index=False).flow.sum()
    history['historical_share']=history.flow/history.groupby('ZCTA').flow.transform('sum').replace(0,np.nan)
    for name,formula in {'distance_only':'flow ~ log_distance + C(Mall)','distance_income':'flow ~ log_distance + log_income:Buckhead + C(Mall)','distance_race':'flow ~ log_distance + black_share:Buckhead + C(Mall)'}.items():
        fit=smf.glm(formula,data=train,family=sm.families.Poisson()).fit(maxiter=120)
        test2=test.copy()
        test2['predicted']=fit.predict(test2)
        test2['pred_share']=test2.predicted/test2.groupby(['ZCTA','Month']).predicted.transform('sum')
        test2['actual_share']=test2.flow/test2.groupby(['ZCTA','Month']).flow.transform('sum').replace(0,np.nan)
        test2=test2.merge(history[['ZCTA','Mall','historical_share']],on=['ZCTA','Mall'],how='left').dropna(subset=['pred_share','actual_share','historical_share'])
        rows.append({'scenario':scenario,'model':name,'model_MAE_pp':100*np.mean(abs(test2.pred_share-test2.actual_share)),'baseline_MAE_pp':100*np.mean(abs(test2.historical_share-test2.actual_share)),'n':len(test2)})
comparison=pd.DataFrame(rows)
print(comparison.to_string(index=False))
comparison.to_csv(PROCESSED/'rerun_holdout.csv',index=False)